### Tools
- Basic chat
- Internet research
- Google maps (estimated time)
- Weather
- Timer 
- AC control
- Globe light controls

In [ ]:
from google import genai
import os 
import json

ai_api_key = os.getenv('GOOGLE_AI_KEY')

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=ai_api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash-lite-preview-06-17", contents="Explain how AI works in a few words"
)
print(response.text)

In [ ]:
import requests
from datetime import datetime, timedelta
import math

HOME_ADDRESS = os.getenv('GOOGLE_API_HOME_ADDRESS')

def get_adress_place_id(address_string: str, maps_key: str) -> str:
    response = requests.get(
        "https://maps.googleapis.com/maps/api/geocode/json", 
        params={'address':address_string, 
        'key':maps_key}
        )
    
    if response.status_code == 200:
        return response.json()['results'][0]['place_id']
    else:
        return "Error: could not get response."

def get_address_long_lat(address_string: str, maps_key: str) -> dict:
    response = requests.get(
        "https://maps.googleapis.com/maps/api/geocode/json", 
        params={'address':address_string, 
        'key':maps_key}
        )
    
    if response.status_code == 200:
        return response.json()['results'][0]['navigation_points'][0]['location']
    else:
        return {'Error':"Error: could not get response."}

def get_route(destination_address: str, maps_key: str, origin_address: str = HOME_ADDRESS) -> dict:
    origin_id = get_adress_place_id(origin_address, maps_key)
    destination_id = get_adress_place_id(destination_address, maps_key)
    
    url = 'https://routes.googleapis.com/directions/v2:computeRoutes'

    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': maps_key,
        'X-Goog-FieldMask': 'routes.duration,routes.staticDuration,routes.distanceMeters,routes.legs.steps.navigationInstruction'
    }

    payload = {
        "origin":{
            "placeId": origin_id
        },
        "destination":{
            "placeId": destination_id
        },
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_AWARE",
        "computeAlternativeRoutes": False,
        "routeModifiers": {
            "avoidTolls": False,
            "avoidHighways": False,
            "avoidFerries": False
        },
        "languageCode": "en-CA",
        "units": "METRIC"
    }

    response = requests.post(url, headers=headers, data=json.dumps(payload))
    if response.status_code == 200:
        response_json = response.json()
        return {'route_steps':response_json['routes'][0]['legs'][0]['steps'], 
                'time_with_traffic':math.ceil(int(response_json['routes'][0]['duration'][:-1])/60),
                'time_without_traffic':math.ceil(int(response_json['routes'][0]['staticDuration'][:-1])/60),
                'distnce':math.ceil(int(response_json['routes'][0]['distanceMeters'])/1000)}
    
    else:
        return {'Error':"Error: could not get response."}

def get_current_weather(maps_key: str, address_string: str = HOME_ADDRESS) -> dict:
    long_lat_dict = get_address_long_lat(address_string, maps_key)

    if 'Error' in long_lat_dict:
        return {'Error':"Error: could not get response."}
    
    params = {'key':maps_key,
              'location.latitude':long_lat_dict['latitude'],
              'location.longitude':long_lat_dict['longitude']}
    response = requests.get("https://weather.googleapis.com/v1/currentConditions:lookup", params=params)
    if response.status_code == 200:
        response_json = response.json()

        keys_to_remove = ['timeZone', 'isDaytime', 'dewPoint', 'airPressure', 'heatIndex', 'windChill', 
                          'currentConditionsHistory', 'uvIndex']

        for key in keys_to_remove:
            response_json.pop(key, None)

        return response_json
    
    else:
        return {'Error':"Error: could not get response."}

def get_daily_weather(maps_key: str, forecast_days: str, address_string: str = HOME_ADDRESS) -> list:
    long_lat_dict = get_address_long_lat(address_string, maps_key)

    if 'Error' in long_lat_dict:
        return {'Error':"Error: could not get response."}
    
    params = {'key':maps_key,
              'location.latitude':long_lat_dict['latitude'],
              'location.longitude':long_lat_dict['longitude'],
              'days':forecast_days,
              'pageSize':forecast_days}
    response = requests.get("https://weather.googleapis.com/v1/forecast/days:lookup", params=params)
    if response.status_code == 200:
        response_json = response.json()

        keys_to_remove = ['interval', 'nighttimeForecast', 'sunEvents', 'moonEvents', 'maxHeatIndex']

        for i in range(len(response_json['forecastDays'])):
            for key in keys_to_remove:
                response_json['forecastDays'][i].pop(key, None)

        response_json.pop('timeZone')
        return response_json['forecastDays']
    
    else:
        return ["Error: could not get response."]

def get_hourly_weather(maps_key: str, address_string: str = HOME_ADDRESS) -> list:
    long_lat_dict = get_address_long_lat(address_string, maps_key)

    if 'Error' in long_lat_dict:
        return {'Error':"Error: could not get response."}
    
    params = {'key':maps_key,
              'location.latitude':long_lat_dict['latitude'],
              'location.longitude':long_lat_dict['longitude'],
              'hours':48,
              'pageSize':48}
    response = requests.get("https://weather.googleapis.com/v1/forecast/hours:lookup", params=params)

    if response.status_code == 200:
        response_json = response.json()
        final_response_list = []
        keys_to_keep = ['displayDateTime', 'weatherCondition', 'temperature', 'feelsLikeTemperature', 'precipitation', 'relativeHumidity']

        for forecast in response_json['forecastHours']:
            if forecast['isDaytime']:
                temp_dict = {}
                for key in forecast:
                    if key in keys_to_keep:
                        temp_dict[key] = forecast[key]
                final_response_list.append(temp_dict)

        return final_response_list
    
    else:
        return ["Error: could not get response."]

address_string = "Le Crux, Boisbriand, QC, Canada"#"933 Place thiboutot, ste-therese, quebec, canada"
maps_key = os.getenv('GOOGLE_MAPS_KEY')

destination_id = get_adress_place_id(address_string, maps_key)
response = get_route(destination_id, maps_key)

#get_current_weather(maps_key)
#get_daily_weather(maps_key, 3)
#get_hourly_weather(maps_key)
